# S3 FX trend — research panel builder

Builds and caches ready-to-load artifacts under `01_data/data_files/s3_fx_trend/`
so hypothesis notebooks do not re-fetch every run.

**Writes**

| File | Builder |
|------|---------|
| `s3_price_panel_1d.parquet` | `build_s3_price_panel(..., interval="1d")` |
| `s3_price_panel_1h.parquet` | `build_s3_price_panel(..., interval="1h")` (optional / OANDA) |
| `economic_calendar.parquet` | `load_economic_calendar(refresh=True)` |
| `s3_event_panel.parquet` | `build_s3_event_panel` |
| `g10_policy_rates.parquet` | `fetch_all_g10_policy_rates` |
| `bis_reer_monthly.parquet` | `fetch_bis_reer` |

**Credentials:** `OANDA_API_TOKEN` + `FRED_API_KEY` in `config/credentials.env`.
Daily prices can fall back to `source="yfinance"` if OANDA is unavailable.


## 0. Imports & Config


In [ ]:
import os
import sys
import warnings

import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from data.ingestion.alternative_data.bis_reer import BIS_REER_SERIES, fetch_bis_reer
from data.ingestion.economic_calendar_fetcher import load_economic_calendar
from data.ingestion.fx_fetcher import G10_V1_PAIRS
from data.ingestion.rates_fetcher import fetch_all_g10_policy_rates
from data.processing.s3_fx_event_panel import build_s3_event_panel
from data.processing.s3_fx_price_panel import (
    RESEARCH_IS_END_S3,
    add_donchian,
    add_realised_vol,
    add_tsmom,
    build_s3_price_panel,
    s3_data_dir,
)

DATA_DIR = s3_data_dir(ROOT)
os.makedirs(DATA_DIR, exist_ok=True)
START = "2007-01-01"
PRICE_SOURCE = os.environ.get("S3_PRICE_SOURCE", "oanda")  # or yfinance
print("DATA_DIR=", DATA_DIR)
print("RESEARCH_IS_END_S3=", RESEARCH_IS_END_S3)
print("G10_V1_PAIRS=", G10_V1_PAIRS)
print("PRICE_SOURCE=", PRICE_SOURCE)


## 1. Daily price panel


In [ ]:
try:
    panel_1d = build_s3_price_panel(
        G10_V1_PAIRS,
        start=START,
        interval="1d",
        source=PRICE_SOURCE,
        cache=True,
        data_dir=DATA_DIR,
    )
except Exception as exc:
    warnings.warn(f"OANDA/primary failed ({exc!r}); retrying source=yfinance")
    panel_1d = build_s3_price_panel(
        G10_V1_PAIRS,
        start=START,
        interval="1d",
        source="yfinance",
        cache=True,
        data_dir=DATA_DIR,
    )

if not panel_1d.empty:
    panel_1d = add_tsmom(panel_1d, 21)
    panel_1d = add_tsmom(panel_1d, 63)
    panel_1d = add_tsmom(panel_1d, 252)
    panel_1d = add_donchian(panel_1d, 55)
    panel_1d = add_realised_vol(panel_1d, 21)
    out = os.path.join(DATA_DIR, "s3_price_panel_1d.parquet")
    panel_1d.to_parquet(out, index=False)
    print("wrote", out, "rows=", len(panel_1d))
else:
    print("empty 1d panel — check credentials / network")


## 2. Hourly price panel (E-003; skip if no OANDA)


In [ ]:
panel_1h = pd.DataFrame()
try:
    panel_1h = build_s3_price_panel(
        G10_V1_PAIRS,
        start=START,
        interval="1h",
        source="oanda",
        cache=True,
        data_dir=DATA_DIR,
    )
    print("1h rows=", len(panel_1h))
except Exception as exc:
    warnings.warn(f"skip 1h panel (OANDA required): {exc!r}")


## 3. Economic calendar + event panel


In [ ]:
cal = load_economic_calendar(refresh=True)
print("calendar rows=", len(cal), "event_ids=", cal["event_id"].nunique() if not cal.empty else 0)
events = build_s3_event_panel(cal, N=20, cache=True, data_dir=DATA_DIR)
print("event panel rows=", len(events))


## 4. G10 policy rates (FRED)


In [ ]:
rates_path = os.path.join(DATA_DIR, "g10_policy_rates.parquet")
try:
    rates = fetch_all_g10_policy_rates(start=START)
    if rates is not None and not rates.empty:
        rates = rates.reset_index()
        rates.to_parquet(rates_path, index=False)
        print("wrote", rates_path, "rows=", len(rates))
    else:
        print("empty rates frame")
except Exception as exc:
    warnings.warn(f"FRED policy rates failed (need FRED_API_KEY): {exc!r}")


## 5. BIS REER monthly (PIT M+2 availability_date)


In [ ]:
reer_path = os.path.join(DATA_DIR, "bis_reer_monthly.parquet")
try:
    ccys = sorted(BIS_REER_SERIES.keys())
    reer = fetch_bis_reer(ccys, start=START)
    if reer is not None and not reer.empty:
        reer = reer.reset_index()
        reer.to_parquet(reer_path, index=False)
        print("wrote", reer_path, "rows=", len(reer))
        print(reer.head())
    else:
        print("empty REER frame")
except Exception as exc:
    warnings.warn(f"BIS REER failed (need FRED_API_KEY): {exc!r}")


## 6. Summary


In [ ]:
for name in (
    "s3_price_panel_1d.parquet",
    "s3_price_panel_1h.parquet",
    "economic_calendar.parquet",
    "s3_event_panel.parquet",
    "g10_policy_rates.parquet",
    "bis_reer_monthly.parquet",
):
    p = os.path.join(DATA_DIR, name)
    print(f"{name}: exists={os.path.isfile(p)} size={os.path.getsize(p) if os.path.isfile(p) else 0}")
